# First Backtest — Interactive Walkthrough

Same logic as `first_backtest.py`, but split into cells so you can **see every DataFrame** as it changes.

**How to use:** click a cell and press `Shift+Enter` to run it. Run them top to bottom.

Tip: the *last expression* in a cell is what gets displayed. So ending a cell with `df.head()` shows a table.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make charts appear inline in the notebook
%matplotlib inline

# Show more rows/columns when we peek at DataFrames
pd.set_option("display.max_columns", 30)
print("ready")

## Step 1 — Get the data

Download daily Bitcoin prices. After it runs, look at the table: each **row is one day**, each **column is a price** (Open/High/Low/Close/Volume).

In [ ]:
import yfinance as yf

df = yf.download("BTC-USD", start="2019-01-01", auto_adjust=True, progress=False)

# yfinance sometimes returns multi-level columns — flatten them
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()

# Show the first 5 rows (this table renders below the cell)
df.head()

In [ ]:
# A few ways to "see" a DataFrame from different angles:
print("Shape (rows, columns):", df.shape)
print("\nColumn data types:")
print(df.dtypes)
print("\nLast 5 rows:")
df.tail()

In [ ]:
# .describe() gives summary statistics for each numeric column
df.describe()

## Step 2 — Build the signal

Add two moving averages and a 0/1 column: **1 = we want to be holding**, **0 = we want to be in cash**.

Notice the first rows of `ma_slow` are `NaN` — you can't average 100 days when only a few days exist yet.

In [ ]:
fast, slow = 20, 100

df["ma_fast"] = df["Close"].rolling(fast).mean()
df["ma_slow"] = df["Close"].rolling(slow).mean()
df["signal"] = (df["ma_fast"] > df["ma_slow"]).astype(int)

# Look at rows AFTER day 100 so the moving averages have values
df[["Close", "ma_fast", "ma_slow", "signal"]].iloc[100:110]

In [ ]:
# See it visually: price with the two moving averages
df[["Close", "ma_fast", "ma_slow"]].plot(figsize=(11, 5), title="Close vs moving averages")
plt.show()

## Step 3 — Backtest it (the honest way)

Two crucial ideas:
- **(A) Anti-lookahead shift:** today's *position* is *yesterday's* signal (`.shift(1)`) — you can only act on a signal the day after you see it.
- **(B) Costs:** every time the position flips (0↔1) you pay a fee.

Run this, then compare the `signal` and `position` columns — `position` is `signal` slid down by one row.

In [ ]:
cost_per_trade = 0.001  # 0.1% per trade

df["ret"] = df["Close"].pct_change()                     # daily return of holding
df["position"] = df["signal"].shift(1).fillna(0)          # (A) the shift
df["trade"] = df["position"].diff().abs().fillna(0)       # 1 on days we switch
df["cost"] = df["trade"] * cost_per_trade                 # (B) costs
df["strat_ret"] = df["position"] * df["ret"] - df["cost"] # strategy daily return

df["equity_strategy"] = (1 + df["strat_ret"]).cumprod()   # compounding equity curve
df["equity_buyhold"] = (1 + df["ret"]).cumprod()

df = df.dropna()

# See the shift in action: position = signal moved down one row
df[["signal", "position", "ret", "trade", "strat_ret"]].iloc[100:110]

In [ ]:
# Just the days where a trade actually happened
df[df["trade"] == 1][["Close", "signal", "position", "cost"]].head(15)

## Step 4 — Measure it

Three numbers that start every quant conversation: total return, Sharpe ratio, max drawdown.

In [ ]:
def metrics(daily_returns, periods_per_year=365):
    r = daily_returns.dropna()
    total_return = (1 + r).prod() - 1
    sharpe = np.sqrt(periods_per_year) * r.mean() / r.std() if r.std() > 0 else 0.0
    equity = (1 + r).cumprod()
    drawdown = equity / equity.cummax() - 1
    return total_return, sharpe, drawdown.min()

# Put the two sets of metrics side by side in a small DataFrame so it's readable
results = pd.DataFrame(
    {
        "Strategy": metrics(df["strat_ret"]),
        "Buy & Hold": metrics(df["ret"]),
    },
    index=["Total return", "Sharpe ratio", "Max drawdown"],
)
results

In [ ]:
print("Number of trades:", int(df["trade"].sum()), "(each one cost you money)")

## Step 5 — See it

The two equity curves, growth of 1 unit of money over time.

In [ ]:
plt.figure(figsize=(11, 6))
plt.plot(df.index, df["equity_buyhold"], label="Buy & Hold", linewidth=1.5)
plt.plot(df.index, df["equity_strategy"], label="MA Crossover Strategy", linewidth=1.5)
plt.title("BTC-USD: Strategy vs Buy & Hold (growth of 1 unit)")
plt.ylabel("Portfolio value (started at 1)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Now experiment

This is how you actually learn. Try changing things and re-running from Step 2:
- Change `fast, slow` to `10, 50` or `50, 200`. Watch the trade count and Sharpe move.
- Change the symbol in Step 1 to `"ETH-USD"`.
- Set `cost_per_trade = 0` and see how much "profit" was actually just fees.

Ask yourself after each run: *am I finding an edge, or fitting noise?*